In [ ]:

import requests
import pandas as pd

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

### **Download and Parse the PDF**

In [ ]:
url = "https://unstats.un.org/unsd/demographic-social/products/worldswomen/annex_tables/Table1A.pdf"
local_file = "Table1A.pdf"

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

pipeline_options = PdfPipelineOptions()
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert(local_file)
doc = res.document

print(f"Tables extracted: {len(doc.tables)}")

In [ ]:
# Collect fragments with page metadata
fragments = []
for table in doc.tables:
    df = table.export_to_dataframe(doc)
    page = table.prov[0].page_no if table.prov else 0
    fragments.append({'df': df, 'page': page})
    print(f"Page {page}: {df.shape[0]:>2} rows × {df.shape[1]} cols")

total_rows = sum(f['df'].shape[0] for f in fragments)
print(f"\nTotal: {total_rows} rows across {len(fragments)} fragments")

In [ ]:
# Column names differ across pages despite being the same logical table
print("Fragment 0 columns (first 4):")
print(f"  {list(fragments[0]['df'].columns[:4])}\n")
print("Fragment 1 columns (first 4):")
print(f"  {list(fragments[1]['df'].columns[:4])}\n")

match = list(fragments[0]['df'].columns) == list(fragments[1]['df'].columns)
print(f"Column names match across fragments: {match}")
print(f"Column count matches: "
      f"{fragments[0]['df'].shape[1]} == {fragments[1]['df'].shape[1]}")

In [ ]:
# First rows of fragment 0 (page 1)
fragments[0]['df'].head(4)

In [ ]:
# First rows of fragment 1 (page 2) — note the "(continued)" marker row
fragments[1]['df'].head(4)

In [ ]:
def has_duplicate_header(df):
    """Check if the first data row duplicates the column names."""
    if len(df) == 0:
        return False
    first_row = [str(v).strip() for v in df.iloc[0]]
    columns = [str(c).strip() for c in df.columns]
    return first_row == columns


def stitch_tables(extracted_tables):
    """
    Stitch consecutive table fragments with matching column counts.

    Parameters
    ----------
    extracted_tables : list of dict
        Each dict has 'df' (DataFrame) and 'page' (int).

    Returns
    -------
    list of DataFrame
        Tables with multi-page fragments merged where appropriate.
    """
    if not extracted_tables:
        return []

    tables = sorted(extracted_tables, key=lambda t: t['page'])
    result = []
    current = tables[0]['df'].copy()
    current_page = tables[0]['page']

    for entry in tables[1:]:
        next_df = entry['df'].copy()
        next_page = entry['page']

        if next_page == current_page + 1 and next_df.shape[1] == current.shape[1]:
            # Check for duplicate header before standardizing column names
            if has_duplicate_header(next_df):
                print(f'  Page {next_page}: removing dup header row')
                next_df = next_df.iloc[1:].reset_index(drop=True)

            # Standardize column names from the first fragment
            next_df.columns = current.columns

            print(f'  Stitching page {current_page} + {next_page}'
                  f' \u2192 {len(current) + len(next_df)} rows')
            current = pd.concat([current, next_df], ignore_index=True)
        else:
            result.append(current)
            current = next_df

        current_page = next_page

    result.append(current)
    return result

In [ ]:
stitched = stitch_tables(fragments)
print(f'\nResult: {len(stitched)} table(s)')

table = stitched[0]
print(f'Shape: {table.shape[0]} rows \u00d7 {table.shape[1]} columns')

In [ ]:
# Assign readable column names (Docling's parsed names are noisy)
COLUMNS = [
    'Country or area',
    'Pop. 1950 Women (000s)', 'Pop. 1950 Men (000s)',
    'Pop. 1980 Women (000s)', 'Pop. 1980 Men (000s)',
    'Pop. 2010 Women (000s)', 'Pop. 2010 Men (000s)',
    'Men per 100 Women',
    'Share 60+ Women (%)', 'Share 60+ Men (%)',
    'TFR 1950-55', 'TFR 1980-85', 'TFR 2005-10',
    'Marriage Age Year', 'Marriage Age Women', 'Marriage Age Men'
]
table.columns = COLUMNS
table.head(10)

In [ ]:
table.tail(10)